In [ ]:
!pip install pymupdf

In [ ]:
import fitz
import re

doc = fitz.open("sample-sdf-document.pdf")
results = []
vendor_pages_seen = []

for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text("text")
    lines = text.split("\n")
    lines = [line.strip() for line in lines if line.strip()]

    for i, line in enumerate(lines):
        if line.strip() == "Cytiva" and page_num not in vendor_pages_seen:
            vendor_pages_seen.append(page_num)
            results.append({
                "page": page_num + 1,
                "label": "Vendor Name",
                "value": "Cytiva"
            })

        if page_num == 0 and re.match(r'\d+\s+\w+,\s+\d{4}', line):
            results.append({
                "page": page_num + 1,
                "label": "Letter Date",
                "value": line.strip()
            })

        matched = False
        for keyword in ["Lot Number", "Date of Manufacture", "Expiration Date"]:
            if line.startswith(keyword) and not matched:
                matched = True
                match = re.search(r'\d+', line)
                value = match.group(0) if match else (lines[i+1].strip() if i+1 < len(lines) else "unknown")
                results.append({
                    "page": page_num + 1,
                    "label": keyword,
                    "value": value
                })

print("=== EXTRACTED DATE FIELDS ===")
for r in results:
    print(f"Page {r['page']} | {r['label']}: {r['value']}")

=== EXTRACTED DATE FIELDS ===
Page 1 | Vendor Name: Cytiva
Page 1 | Letter Date: 3 June, 2022
Page 2 | Lot Number: 17242818
Page 2 | Date of Manufacture: 20210126
Page 2 | Expiration Date: 20230126
Page 3 | Vendor Name: Cytiva
Page 3 | Lot Number: 18356721
Page 3 | Date of Manufacture: 20240315
Page 3 | Expiration Date: 20260315
